<a href="https://colab.research.google.com/github/shweta-1202/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shweta-1202/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one webpage. The search performance information includes a 90-day observation window, such as impressions over the last 90 days. Other fields describe the page itself, such as content age and days since the last update.

In [2]:
import os
import subprocess
import pandas as pd

# Download the starter repository if it is not already available
if not os.path.exists("flyrank-ml-internship-starter"):
    subprocess.run([
        "git", "clone",
        "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
    ], check=True)

df = pd.read_csv(
    "flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"
)

print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

df.head()

Number of rows: 30000
Number of columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

I plan to use content_age_days, days_since_last_update, impressions_90d, avg_position, ctr, word_count, and search_volume as possible features.

### Label

The label is whether the page is declining. I will create is_declining from trend_direction, where down means 1 and other directions mean 0.

### Context

content_type and trend_direction can be used to understand and describe the data.

### Excluded

I will exclude trend_pct from the model features because it is directly related to the trend outcome and could cause data leakage. I will also avoid using any information that would only be known after the decision or outcome.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
df["is_declining"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nMissing values:")
print(df.isna().sum())

print("\nTrend direction:")
print(df["trend_direction"].value_counts())

print("\nDeclining pages:", df["is_declining"].sum())

Rows: 30000
Columns: 45

Missing values:
content_id                    0
client_id                     0
search_volume              2468
competition                2468
competition_level          2610
cpc                        2468
content_type                  0
main_intent                2374
word_count                 7699
char_count                 7699
provider_used             21438
model_used                 5733
impressions_90d               0
clicks_90d                    0
pageviews_90d                 0
sessions_90d                  0
users_90d                     0
engaged_sessions_90d          0
ai_sessions_90d               0
scroll_events_90d             0
days_with_impressions         0
days_with_sessions            0
impressions_last_30d          0
clicks_last_30d               0
sessions_last_30d             0
impressions_prev_30d          0
clicks_prev_30d               0
sessions_prev_30d             0
content_age_days              0
age_tier                      0

In [4]:
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "search_volume"
]

print(df[features].describe())

       content_age_days  days_since_last_update  impressions_90d  \
count       30000.00000            30000.000000     30000.000000   
mean          256.16780               46.098300      5200.366300   
std           132.70793               42.078709     16838.019547   
min            90.00000                1.000000         1.000000   
25%           132.00000               20.000000        81.000000   
50%           236.00000               20.000000       731.000000   
75%           333.00000              104.000000      3615.250000   
max           564.00000              373.000000    517715.000000   

       avg_position           ctr    word_count  search_volume  
count   30000.00000  30000.000000  22301.000000   27532.000000  
mean       16.34238      0.510733   3107.760325     158.882391  
std        15.21679      3.279162   1452.382598    1518.270825  
min         0.00000      0.000000      8.000000       0.000000  
25%         6.20000      0.000000   2413.000000       0.000000

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset has some limits. It can show patterns and relationships in the available data, but it cannot prove that one factor caused a page to decline. It also cannot tell us how Google's ranking algorithm works or predict exactly what Google will do in the future. The data covers a limited observation period, so the results may not represent every possible situation. I will use the results as measured and directional decision-support information rather than causal proof.

In [5]:
print("Impressions column:", "impressions_90d" in df.columns)
print("Number of pages:", len(df))
print("Missing impressions:", df["impressions_90d"].isna().sum())

Impressions column: True
Number of pages: 30000
Missing impressions: 0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.